# DysonianLineCNN — Infer on a real experimental spectrum

Thin wrapper around [`dyson_cnn.infer.predict_real_spectrum`](../dyson_cnn/infer.py).

**Before running**:

1. A trained run exists in `runs/` (use [`01_train_and_eval.ipynb`](01_train_and_eval.ipynb)
   if you do not have one yet).
2. `config/inference.json.runName` points at it.
3. `matlab/PrepareOneSpectrumForCNN.m` has been run to produce
   `<spectrum_basename>_spectrum.csv` in the project folder.

On Colab: add `GITHUB_DEPLOY_KEY` to Colab Secrets before first run. See
[README.md](../README.md) 'Colab setup'.

## 1. Environment setup

Idempotent across Google Colab and local Mac.

**On Colab** (`google.colab` module detected):
1. Read the SSH deploy key from Colab Secrets (`GITHUB_DEPLOY_KEY`) and write it
   to `~/.ssh/id_ed25519`. The key must be added once per Colab account via the
   key icon in the sidebar — see [README.md](../README.md) "Colab setup".
2. Clone (or pull) `git@github.com:AndriiUriadov/DysonianLineCNN.git` into
   `/content/DysonianLineCNN`.
3. `pip install -e ".[dev]"` the package in editable mode so imports work from
   any cwd and dev dependencies (pytest, nbstripout) are available.
4. Mount Google Drive at `/content/drive`.

**On local Mac**: skip the Colab bootstrap; just verify that the notebook is
running from the repository root so `import dyson_cnn` resolves.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # --- Colab-only: deploy key -> git clone -> pip install -> drive mount ---
    from google.colab import userdata

    ssh_dir = os.path.expanduser("~/.ssh")
    os.makedirs(ssh_dir, exist_ok=True)
    key_path = os.path.join(ssh_dir, "id_ed25519")

    try:
        key_content = userdata.get("GITHUB_DEPLOY_KEY")
    except Exception as e:
        raise RuntimeError(
            "GITHUB_DEPLOY_KEY not found in Colab Secrets. Add it via the key icon "
            "in the left sidebar (see README.md 'Colab setup' section)."
        ) from e

    # Some clipboard paths (e.g., pasting into Colab Secrets through certain
    # browsers on macOS) silently convert LF to CRLF. OpenSSH's libcrypto
    # rejects \r inside the base64 body with "error in libcrypto", so we
    # normalize before writing.
    key_content = key_content.replace("\r\n", "\n").replace("\r", "\n")
    if not key_content.endswith("\n"):
        key_content += "\n"
    with open(key_path, "w") as f:
        f.write(key_content)
    os.chmod(key_path, 0o600)

    # Trust github.com so ssh does not prompt
    get_ipython().system('ssh-keyscan -t ed25519 github.com >> ~/.ssh/known_hosts 2>/dev/null')

    repo_dir = '/content/DysonianLineCNN'
    if not os.path.isdir(repo_dir):
        get_ipython().system('git clone git@github.com:AndriiUriadov/DysonianLineCNN.git ' + repo_dir)
    else:
        get_ipython().system('cd ' + repo_dir + ' && git pull')

    os.chdir(repo_dir)
    get_ipython().system('pip install -q -e . --no-deps')

    from google.colab import drive
    drive.mount('/content/drive')

# Verify we are at the repo root (works on both Colab and local Mac)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dyson_cnn').is_dir():
    for parent in Path.cwd().parents:
        if (parent / 'dyson_cnn').is_dir():
            REPO_ROOT = parent
            os.chdir(REPO_ROOT)
            break
    else:
        raise RuntimeError(f'Could not find repo root from {Path.cwd()}')

print(f'Environment : {"Colab" if IN_COLAB else "local Mac"}')
print(f'Repo root   : {REPO_ROOT}')

## 2. Load configs and resolve paths

In [ ]:
from dyson_cnn.config import load_paths, load_inference_cfg

paths = load_paths(REPO_ROOT / 'config')
inference_cfg = load_inference_cfg(REPO_ROOT / 'config')

run_name = inference_cfg['runName']
spectrum_basename = inference_cfg['spectrum_basename']

run_dir = Path(paths['runs_dir']) / run_name
spectrum_csv = Path(paths['project_dir']) / f'{spectrum_basename}_spectrum.csv'
out_json = Path(paths['project_dir']) / f'{spectrum_basename}_real_predicted_params.json'
out_png  = Path(paths['project_dir']) / 'real_input_channels_preview.png'

print(f'runName        : {run_name}')
print(f'spectrum base  : {spectrum_basename}')
print(f'run_dir        : {run_dir}')
print(f'spectrum_csv   : {spectrum_csv}')

if run_name == 'TO_BE_SET_AFTER_FIRST_TRAINING':
    raise RuntimeError(
        'config/inference.json.runName is still the placeholder. '
        'Train a model with 01_train_and_eval.ipynb and update runName.'
    )
if not run_dir.is_dir():
    raise RuntimeError(f'Run directory not found: {run_dir}')
if not spectrum_csv.exists():
    raise RuntimeError(
        f'Spectrum CSV not found: {spectrum_csv}. '
        f'Run matlab/PrepareOneSpectrumForCNN.m to produce it first.'
    )

## 3. Predict

`predict_real_spectrum` builds the 3-channel input using the **same**
`make_three_channel_input` function that training used — no duplication — then loads the
run's model and denormalizes the output back to Gauss (physical units).

In [ ]:
from dyson_cnn.infer import predict_real_spectrum

result = predict_real_spectrum(
    run_dir=run_dir,
    spectrum_csv_path=spectrum_csv,
    out_json_path=out_json,
    out_preview_png_path=out_png,
)

print('\nPredicted parameters (physical units):')
for k, v in result['y_pred_physical'].items():
    print(f'  {k}: {v:.6g}')
print(f'\nSaved JSON: {out_json}')
print(f'Saved PNG : {out_png}')

## 4. Next step

Run [`matlab/Validator.m`](../matlab/Validator.m) to reconstruct the Dysonian curve from
the predicted `B0`, `dB`, `p3` parameters and overlay it on the raw experimental spectrum
for visual quality assessment.